# Stage 1 — EDGAR 8-K Pull

Pulls **Item 2.02** (earnings release) 8-K filings for S&P 500 companies from the SEC EDGAR API.

**Output**
- `data/filings_raw.csv` — one row per filing with metadata + body text
- `data/filings_index.csv` — lightweight index (no text) for quick inspection
- `data/pull.log` — timestamped run log
- `data/checkpoint.json` — resume state (re-run safely after interruption)

**Requirements**
```
pip install requests pandas beautifulsoup4 lxml tqdm
```

## 0. Install dependencies

## 1. Configuration

> ⚠️ **Update `EDGAR_USER_AGENT`** with your real name and email before running.  
> The SEC requires a valid User-Agent and will block anonymous requests.

In [ ]:
import requests
import pandas as pd
import json
import time
import re
import logging
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

# ── Update this before running ────────────────────────────────────────────────
EDGAR_USER_AGENT = "Jargalsaikhan Gansuld gansuld_jargalsaikha@student.ceu.edu"

# ── Date range ────────────────────────────────────────────────────────────────
START_DATE = "2023-11-01"
END_DATE   = "2026-04-30"

# ── Item filter ───────────────────────────────────────────────────────────────
TARGET_ITEM = "2.02"   # Results of Operations / Earnings Releases

# ── Rate limiting (EDGAR max is 10 req/s — stay comfortably under) ────────────
REQUEST_DELAY = 0.12   # seconds between requests
MAX_RETRIES   = 3

# ── Output paths ─────────────────────────────────────────────────────────────
OUTPUT_DIR      = Path("data")
OUTPUT_PATH     = OUTPUT_DIR / "filings_raw.csv"
INDEX_PATH      = OUTPUT_DIR / "filings_index.csv"
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint.json"
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[
        logging.FileHandler(OUTPUT_DIR / "pull.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

HEADERS = {"User-Agent": EDGAR_USER_AGENT}

print("Configuration loaded.")

## 2. Helper — HTTP with retries

In [ ]:
def _get(url: str, retries: int = MAX_RETRIES) -> requests.Response:
    """GET with automatic retry on 5xx / connection errors."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            if resp.status_code == 200:
                time.sleep(REQUEST_DELAY)
                return resp
            elif resp.status_code == 429:
                log.warning("Rate limited — sleeping 60s...")
                time.sleep(60)
            elif resp.status_code >= 500:
                log.warning(f"Server error {resp.status_code}, retry {attempt + 1}")
                time.sleep(2 ** attempt)
            else:
                resp.raise_for_status()
        except requests.exceptions.RequestException as e:
            log.warning(f"Request failed ({e}), retry {attempt + 1}")
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed to fetch {url} after {retries} retries")

## 3. Get S&P 500 tickers

In [ ]:
# ── Stage 1, Section 3: Get historical S&P 500 universe (survivorship-bias-corrected) ─────

from io import StringIO

def get_historical_sp500_tickers(
    start_date: str = "2022-01-01",
    end_date:   str = "2024-12-31",
) -> list[str]:
    """
    Returns all unique tickers that were S&P 500 members at ANY point
    during [start_date, end_date], correcting for survivorship bias.

    Strategy
    --------
    1. Fetch the current constituent list from Wikipedia (Table 0).
    2. Fetch the historical changes table (Table 1).
    3. Add back tickers REMOVED between start_date and end_date —
       they were members during our study window.
    4. Remove tickers ADDED after end_date — they appear on today's
       Wikipedia page but were not in the index during our window.

    Note: Companies removed *before* start_date are already absent from
    both the current list and step 3, so they are naturally excluded.
    """
    start = pd.Timestamp(start_date)
    end   = pd.Timestamp(end_date)

    url      = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    tables   = pd.read_html(StringIO(response.text), flavor="lxml")

    # ── Step 1: current constituents ──────────────────────────────────────────
    current_df      = tables[0]
    current_tickers = set(s.replace(".", "-") for s in current_df["Symbol"].tolist())
    log.info(f"Current S&P 500 constituents: {len(current_tickers)}")

    # ── Step 2: parse the changes table ───────────────────────────────────────
    # Wikipedia's changes table has multi-level column headers that pandas
    # returns as tuples, e.g. ('Added', 'Ticker'), ('Removed', 'Ticker').
    # Flatten them to lowercase strings for robust matching.
    changes_df = tables[1].copy()
    changes_df.columns = [
        "_".join(str(c).strip().lower() for c in col).strip("_")
        if isinstance(col, tuple) else str(col).strip().lower()
        for col in changes_df.columns
    ]

    # Locate the date column (always first)
    date_col = changes_df.columns[0]
    changes_df[date_col] = pd.to_datetime(changes_df[date_col], errors="coerce")
    changes_df = changes_df.dropna(subset=[date_col])

    # Identify added/removed ticker columns by substring matching
    added_col   = next((c for c in changes_df.columns if "added"   in c and "ticker" in c), None)
    removed_col = next((c for c in changes_df.columns if "removed" in c and "ticker" in c), None)

    if added_col is None or removed_col is None:
        log.warning(
            "Could not parse changes table columns. "
            f"Found: {list(changes_df.columns)}. "
            "Falling back to current constituent list only — survivorship bias NOT corrected."
        )
        return sorted(current_tickers)

    def clean_ticker(t) -> str | None:
        """Normalise a ticker string; return None if unusable."""
        if not isinstance(t, str) or t.strip() in ("", "—", "-", "nan"):
            return None
        return t.strip().replace(".", "-").upper()

    # ── Step 3: add back tickers removed during [start_date, end_date] ───────
    # A company removed on or after start_date was a member at the beginning
    # of our study window (or entered and exited during it).
    mask_removed_in_window = changes_df[date_col] >= start
    removed_tickers = set(
        ct for t in changes_df.loc[mask_removed_in_window, removed_col]
        if (ct := clean_ticker(t)) is not None
    )
    log.info(
        f"Tickers removed from S&P 500 on/after {start_date}: "
        f"{len(removed_tickers)} — adding back to universe."
    )

    # ── Step 4: remove tickers added AFTER end_date ───────────────────────────
    # These appear on today's Wikipedia page but were not in the index
    # during 2022–2024.
    mask_added_after_end = changes_df[date_col] > end
    late_additions = set(
        ct for t in changes_df.loc[mask_added_after_end, added_col]
        if (ct := clean_ticker(t)) is not None
    )
    log.info(
        f"Tickers added to S&P 500 after {end_date}: "
        f"{len(late_additions)} — removing from universe."
    )

    # ── Combine ───────────────────────────────────────────────────────────────
    universe = (current_tickers | removed_tickers) - late_additions
    log.info(
        f"Historical universe ({start_date} to {end_date}): "
        f"{len(universe)} unique tickers "
        f"(current={len(current_tickers)}, "
        f"added_back={len(removed_tickers - current_tickers)}, "
        f"dropped={len(late_additions)})"
    )
    return sorted(universe)


# ── Run it ────────────────────────────────────────────────────────────────────
sp500_tickers = get_historical_sp500_tickers(
    start_date=START_DATE,
    end_date=END_DATE,
)
print(f"Historical S&P 500 universe: {len(sp500_tickers)} tickers")
print("First 10:", sp500_tickers[:10])

In [ ]:
current_set = set(pd.read_html(
    __import__('io').StringIO(
        __import__('requests').get(
            "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
            headers={"User-Agent": "Mozilla/5.0"}
        ).text
    ), flavor="lxml"
)[0]["Symbol"].str.replace(".", "-"))

added_back = sorted(set(sp500_tickers) - current_set)
print(f"Tickers added back ({len(added_back)}):")
print(added_back)

## 4. Map tickers → CIKs

In [ ]:
def get_ticker_to_cik_map() -> dict[str, str]:
    """Downloads EDGAR's official ticker→CIK mapping. CIKs are zero-padded to 10 digits."""
    url  = "https://www.sec.gov/files/company_tickers.json"
    resp = _get(url)
    data = resp.json()
    mapping = {}
    for entry in data.values():
        ticker = entry["ticker"].upper().replace(".", "-")
        cik    = str(entry["cik_str"]).zfill(10)
        mapping[ticker] = cik
    return mapping

ticker_cik_map = get_ticker_to_cik_map()

# Match S&P 500 tickers to CIKs
valid_pairs = [(t, ticker_cik_map[t]) for t in sp500_tickers if t in ticker_cik_map]
missing     = [t for t in sp500_tickers if t not in ticker_cik_map]

print(f"Matched : {len(valid_pairs)} tickers")
print(f"Missing : {len(missing)} tickers (will be skipped)")
if missing:
    print("Missing tickers:", missing)

In [ ]:
# ── Section 4b: Recover CIKs for tickers missing from company_tickers.json ───
# These are typically delisted, acquired, or ticker-changed firms.
# We search EDGAR's company search API by ticker symbol as a fallback.

def search_cik_by_ticker(ticker: str) -> str | None:
    """
    Searches EDGAR's company search endpoint by ticker.
    Returns zero-padded CIK string or None if not found.
    """
    url = f"https://efts.sec.gov/LATEST/search-index?q=%22{ticker}%22&forms=8-K"
    # Use EDGAR's direct company lookup instead
    lookup_url = f"https://www.sec.gov/cgi-bin/browse-edgar?company=&CIK={ticker}&type=8-K&dateb=&owner=include&count=10&search_text=&action=getcompany&output=atom"
    try:
        resp = _get(lookup_url)
        # CIK appears in the atom feed as /cgi-bin/browse-edgar?action=getcompany&CIK=XXXXXXXXXX
        m = re.search(r"CIK=(\d{10})", resp.text)
        if m:
            return m.group(1)
        # Also try shorter CIK format
        m = re.search(r"CIK=(\d+)", resp.text)
        if m:
            return str(m.group(1)).zfill(10)
    except Exception as e:
        log.warning(f"EDGAR search failed for {ticker}: {e}")
    return None


# Known manual overrides — verified CIKs for acquired/delisted firms
# These were confirmed against EDGAR's company pages directly
MANUAL_CIK_OVERRIDES = {
    "SIVB":  "0000719739",   # SVB Financial Group (collapsed Mar 2023)
    "SBNY":  "0001288946",   # Signature Bank (collapsed Mar 2023)
    "FRC":   "0001026214",   # First Republic Bank (collapsed May 2023)
    "TWTR":  "0001418091",   # Twitter Inc (taken private Oct 2022)
    "ATVI":  "0000718877",   # Activision Blizzard (acquired by MSFT Jan 2023)
    "CERN":  "0000804212",   # Cerner Corp (acquired by Oracle Jun 2022)
    "CTXS":  "0000877890",   # Citrix Systems (taken private Sep 2022)
    "XLNX":  "0000743988",   # Xilinx (acquired by AMD Feb 2022)
    "PXD":   "0001038357",   # Pioneer Natural Resources (acquired by Exxon Oct 2023)
    "PBCT":  "0000763901",   # People's United Financial (acquired by M&T Apr 2022)
    "DRE":   "0000783280",   # Duke Realty (acquired by Prologis Oct 2022)
    "NLSN":  "0001540159",   # Nielsen Holdings (taken private Oct 2022)
    "ABMD":  "0000815787",   # Abiomed (acquired by J&J Jan 2023)
    "DISH":  "0001001082",   # Dish Network (merged with EchoStar)
    "INFO":  "0001065696",   # IHS Markit (merged with S&P Global Feb 2022)
    "DISCA": "0001437107",   # Discovery Inc (merged into WBD Apr 2022)
    "DISCK": "0001437107",   # Discovery Inc class C (same CIK as DISCA)
    "ANSS":  "0000820081",   # Ansys (acquired by Synopsys 2024)
    "CTLT":  "0001584547",   # Catalent (acquired by Novo Holdings 2024)
    "JNPR":  "0001043604",   # Juniper Networks (acquired by HPE 2024)
    "DFS":   "0001393612",   # Discover Financial (acquired by Capital One 2024)
    "FBHS":  "0001519061",   # Fortune Brands (split — check filings)
    "MRO":   "0000101778",   # Marathon Oil (acquired by ConocoPhillips 2024)
    "DAY":   "0001725057"   # Ceredian HCM / Dayforce Inc
}

# Attempt recovery for all missing tickers
recovered = {}
still_missing = []

for ticker in missing:
    # First: check manual overrides
    if ticker in MANUAL_CIK_OVERRIDES:
        recovered[ticker] = MANUAL_CIK_OVERRIDES[ticker]
        log.info(f"Manual override: {ticker} → CIK {MANUAL_CIK_OVERRIDES[ticker]}")
        continue

    # Second: try live EDGAR search
    cik = search_cik_by_ticker(ticker)
    if cik:
        recovered[ticker] = cik
        log.info(f"EDGAR search: {ticker} → CIK {cik}")
    else:
        still_missing.append(ticker)
        log.warning(f"Could not recover CIK for {ticker} — will be skipped")

# Merge recovered CIKs into the valid_pairs list
recovered_pairs = [(t, cik) for t, cik in recovered.items()]
valid_pairs = valid_pairs + recovered_pairs

print(f"\nOriginal matched  : {len(valid_pairs) - len(recovered_pairs)}")
print(f"Recovered via map : {len(recovered_pairs)}")
print(f"Still missing     : {len(still_missing)} {still_missing}")
print(f"Total universe    : {len(valid_pairs)}")

In [ ]:
# Deduplicate valid_pairs by CIK (keeps first occurrence)
seen_ciks = set()
valid_pairs_deduped = []
for ticker, cik in valid_pairs:
    if cik not in seen_ciks:
        valid_pairs_deduped.append((ticker, cik))
        seen_ciks.add(cik)

print(f"Before dedup : {len(valid_pairs)}")
print(f"After dedup  : {len(valid_pairs_deduped)}")
valid_pairs = valid_pairs_deduped

## 5. Fetch 8-K filing list per company

In [ ]:
def get_8k_filings_for_cik(cik: str, start: str, end: str) -> list[dict]:
    """
    Queries the EDGAR submissions API for a CIK.
    Returns all 8-K filings within [start, end] as a list of dicts.
    """
    url  = f"https://data.sec.gov/submissions/CIK{cik}.json"
    resp = _get(url)
    data = resp.json()

    filings     = data.get("filings", {}).get("recent", {})
    forms       = filings.get("form",            [])
    dates       = filings.get("filingDate",       [])
    accessions  = filings.get("accessionNumber",  [])
    primary_docs= filings.get("primaryDocument",  [])

    results = []
    for form, date, acc, doc in zip(forms, dates, accessions, primary_docs):
        if form == "8-K" and (start <= date <= end):
            results.append({
                "accessionNumber": acc,
                "filingDate":      date,
                "primaryDocument": doc,
            })

    # Handle companies with >1000 filings (paginated older results)
    for page in data.get("filings", {}).get("files", []):
        try:
            page_resp = _get(f"https://data.sec.gov/submissions/{page['name']}")
            page_data = page_resp.json()
            for form, date, acc, doc in zip(
                page_data.get("form", []),
                page_data.get("filingDate", []),
                page_data.get("accessionNumber", []),
                page_data.get("primaryDocument", []),
            ):
                if form == "8-K" and (start <= date <= end):
                    results.append({"accessionNumber": acc, "filingDate": date, "primaryDocument": doc})
        except Exception as e:
            log.warning(f"Could not fetch older page {page['name']}: {e}")

    return results

## 6. Download and extract Item 2.02 text

In [ ]:
def fetch_filing_text(cik: str, accession: str, primary_doc: str) -> str | None:
    """Downloads the primary filing document and returns cleaned plain text."""
    acc_nodash = accession.replace("-", "")
    url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_nodash}/{primary_doc}"
    try:
        resp = _get(url)
    except Exception as e:
        log.warning(f"Failed to fetch {url}: {e}")
        return None

    content_type = resp.headers.get("Content-Type", "")
    if "html" in content_type or primary_doc.lower().endswith((".htm", ".html")):
        soup = BeautifulSoup(resp.text, "lxml")
        for tag in soup(["script", "style", "head"]):
            tag.decompose()
        text = soup.get_text(separator=" ", strip=True)
    else:
        text = resp.text

    return text


def contains_item_202(text: str) -> bool:
    """Returns True if the filing contains a reference to Item 2.02."""
    return bool(re.search(r"item\s*2\.02", text, re.IGNORECASE))


def extract_item_202_body(text: str) -> str:
    """
    Extracts the Item 2.02 section body.
    Falls back to full text if the section boundary cannot be found.
    """
    text      = re.sub(r"\s+", " ", text)  # normalise whitespace
    start_pat = re.compile(r"item\s*2\.02", re.IGNORECASE)
    end_pat   = re.compile(r"item\s*\d+\.\d+", re.IGNORECASE)

    m_start = start_pat.search(text)
    if not m_start:
        return text

    m_end    = end_pat.search(text, m_start.end() + 10)
    body_end = m_end.start() if m_end else len(text)
    extracted = text[m_start.start():body_end].strip()

    return extracted if len(extracted) >= 100 else text

## 7. Get precise filing datetime

In [ ]:
def get_filing_datetime(cik: str, accession: str) -> str | None:
    """
    Extracts the precise filing datetime (to the second) from the SGML header.
    Returns ISO format string (e.g. '2023-04-27T16:05:22') or None.
    This is what you use for after-hours / next-open event window logic.
    """
    acc_nodash  = accession.replace("-", "")
    header_url  = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_nodash}/{accession}.txt"
    try:
        resp = _get(header_url)
        m    = re.search(r"<ACCEPTANCE-DATETIME>(\d{14})", resp.text)
        if m:
            raw = m.group(1)  # YYYYMMDDHHMMSS
            return f"{raw[0:4]}-{raw[4:6]}-{raw[6:8]}T{raw[8:10]}:{raw[10:12]}:{raw[12:14]}"
    except Exception:
        pass
    return None

## 8. Checkpoint helpers

In [ ]:
def load_checkpoint() -> set[str]:
    """Returns set of already-processed tickers (for safe resume)."""
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            return set(json.load(f).get("completed", []))
    return set()


def save_checkpoint(completed: set[str]):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"completed": list(completed)}, f)


def save_results(rows: list[dict]):
    """Saves the full dataset and a text-free index CSV."""
    if not rows:
        return
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_PATH, index=False)
    df.drop(columns=["bodyText"], errors="ignore").to_csv(INDEX_PATH, index=False)
    log.info(f"Saved {len(df)} rows → {OUTPUT_PATH}")

## 9. Run the pipeline

Processes all S&P 500 companies, saves incrementally every 10 companies.  
Safe to interrupt and re-run — completed tickers are skipped via the checkpoint file.

In [ ]:
# Load existing results if resuming
if OUTPUT_PATH.exists():
    rows = pd.read_csv(OUTPUT_PATH).to_dict("records")
    print(f"Resuming — {len(rows)} filings already saved.")
else:
    rows = []

completed = load_checkpoint()
print(f"Companies already processed: {len(completed)}")
print(f"Companies remaining        : {len(valid_pairs) - len(completed)}")

In [ ]:
for ticker, cik in tqdm(valid_pairs, desc="Companies", unit="co"):

    if ticker in completed:
        continue

    # Fetch filing list for this company
    try:
        filings = get_8k_filings_for_cik(cik, START_DATE, END_DATE)
    except Exception as e:
        log.error(f"{ticker} ({cik}): filing list failed — {e}")
        completed.add(ticker)
        save_checkpoint(completed)
        continue

    if not filings:
        completed.add(ticker)
        save_checkpoint(completed)
        continue

    log.info(f"{ticker}: {len(filings)} 8-K filings in range")

    for filing in filings:
        acc  = filing["accessionNumber"]
        date = filing["filingDate"]
        doc  = filing["primaryDocument"]

        try:
            text = fetch_filing_text(cik, acc, doc)
        except Exception as e:
            log.warning(f"  {ticker} {acc}: text fetch failed — {e}")
            continue

        if text is None or not contains_item_202(text):
            continue

        body       = extract_item_202_body(text)
        filing_dt  = get_filing_datetime(cik, acc)

        rows.append({
            "ticker":          ticker,
            "cik":             cik,
            "accessionNumber": acc,
            "filingDate":      date,
            "filingDatetime":  filing_dt,
            "primaryDocument": doc,
            "bodyText":        body,
            "bodyTextLen":     len(body),
        })

    completed.add(ticker)
    save_checkpoint(completed)

    # Incremental save every 10 companies
    if len(completed) % 10 == 0:
        save_results(rows)

# Final save
save_results(rows)
print(f"\nDone. {len(rows)} Item 2.02 filings saved to {OUTPUT_PATH}")

## 10. Quick validation

In [ ]:
df = pd.read_csv(OUTPUT_PATH)

print("=" * 50)
print(f"Total filings        : {len(df)}")
print(f"Unique companies     : {df['ticker'].nunique()}")
print(f"Date range           : {df['filingDate'].min()} → {df['filingDate'].max()}")
print(f"Has datetime         : {df['filingDatetime'].notna().sum()} / {len(df)}")
print(f"Median body text len : {df['bodyTextLen'].median():.0f} chars")
print("\nFilings per year:")
print(df['filingDate'].str[:4].value_counts().sort_index())
print("\nTop 10 companies by filing count:")
print(df['ticker'].value_counts().head(10))

In [ ]:
# Preview a sample filing body
sample = df.sample(1).iloc[0]
print(f"Ticker  : {sample['ticker']}")
print(f"Date    : {sample['filingDate']}")
print(f"Datetime: {sample['filingDatetime']}")
print(f"Length  : {sample['bodyTextLen']} chars")
print("\n--- Body text (first 1000 chars) ---")
print(str(sample['bodyText'])[:1000])

In [ ]:
watchlist = ['SIVB', 'FRC', 'SBNY', 'TWTR']
print(df[df['ticker'].isin(watchlist)][['ticker', 'filingDate']].sort_values(['ticker', 'filingDate']))

In [ ]:
frc_late = df[(df['ticker'] == 'FRC') & (df['filingDate'] > '2023-05-01')]
print(frc_late[['ticker', 'filingDate', 'accessionNumber', 'primaryDocument']].to_string())

In [ ]:
before = len(df)

# Drop all FRC rows — they are all Freddie Mac filings under a wrong CIK
df = df[df['ticker'] != 'FRC'].reset_index(drop=True)
print(f"Dropped {before - len(df)} Freddie Mac filings mislabeled as FRC")
print(f"Remaining filings: {len(df)}")

df.to_csv("data/filings_raw.csv", index=False)

In [ ]:
print(df[df['ticker'] == 'SBNY'][['ticker', 'filingDate', 'primaryDocument']].to_string())

In [ ]:
import pandas as pd
df = pd.read_csv("data/filings_raw.csv")
print(len(df))
print(df['filingDate'].min(), df['filingDate'].max())
print(df['filingDate'].str[:7].value_counts().sort_index())